<img src="../img/logo.jpeg" width= 200px>   

# 📓 Notebook 0 - Agentes IA con LangChain     

En este notebook se muestra paso a paso como crear un agente sencillo, para el que se implementarán varias *tools* simples que le añadirán funcionalidades enfocadas a resolver tareas específicas.   
Como "motor" de este agente se usará un modelo de lenguaje largo (LLM) de la gama ***QWEN-3***, en una de las versiones disponible dentro de **Hugging Face**.     

NOTA: Dado que **Hugging Face** a pasado a modalidad de pago por uso, utilizaremos **GROQ** como proveedor de servicios, basando nuestra consultas en el LLM ***GPT-OSS*** de Open AI

La implementación se realiza paso a paso, para que resulte comprensible el proceso de implementación de este tipo de herramientas de IA.   

¡Nos ponemos manos a la obra!   

------

## 🗂️ Índice   
1.- ⚙️ Pasos previos.     
2.- 🔀 Importar librerías.      
3.- 🤖 Nuestro primer agente.          
4.- 🔧 Definir e implementar nuestra primera `TOOL`  - (Patrón `Tool Use`).         
5.- 🧩 "Conectar" con el LLM que se utilizará como "motor" de nuestro Agente.          
6.- 🔍 Lanzar las consultas.             
7.- 👥💬 Configurando el historial de conversaciones.      

-------

## 1.- ⚙️ Pasos previos.

### 🔑 Configuración de las API keys necesarias.    

Lo primero que vamos a hacer es lanzar el siguiente bloque de código para cargar las API keys que necesitamos para este notebook

In [ ]:
# =====================================================================================
# 🔑  Introduce tus API keys en la linea correspondiente antes de ejecutar el notebook
# =====================================================================================
import os


# Carga tu API key de Hugging Face en la siguiente variable. 
# Puedes obtenerla en https://huggingface.co/settings/tokens

HF_API_KEY = ""  # 👈 Pon aquí tu token de HuggingFace

# Carga tu API key de GROQ en la siguiente variable. 
# Puedes obtenerla en https://console.groq.com/keys

GROQ_API_KEY = ""  # 👈 Pon aquí tu token de GROQ

# Configura la variable de entorno para Hugging Face
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ ¡API keys configuradas y listas para usar!")

✅ ¡API keys configuradas y listas para usar!


### 📚 Instalación de librerias



El primer paso a ejecutar es la instalación de las librerías necesarias para poder trabajar en la implementación de nuestro agente.     

Las librerías a instalar son:   
- langchain-openai,
- langchain-huggingface,
- langchain-comunity,
- duckduckgo-search,
- langgraph,
- langchain,
- ddgs,
- sympy.

In [2]:
%pip install langchain-openai langchain-community duckduckgo-search langgraph ddgs langchain-huggingface langchain sympy

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached duckduckgo_search-8.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.14.2-py3-none-any.whl.metadata (137 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx2-2.16.0-py3-none-any.whl.metadata (10 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached websockets-17.4-cp314-cp314-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.3 kB)
  Usi

In [7]:
%pip install -U langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-groq]m [langchain-groq]
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install ipywidgets
# es posible que se necesite ejecutar desde el terminal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]
Note: you may need to restart the kernel to use updated packages.


## 2.- 🔀 Importar librerías

A continuación, importamos los elementos necesarios.

In [10]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import ipywidgets as widgets


from IPython.display import display, Code
from langchain.agents import create_agent

from langchain_groq import ChatGroq

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

from sympy import sympify

Seguidamente, configuramos el motor de búsqueda explicitando cúal se va a utilizar, el idioma y número máximo de resultados del buscador que queremos que nos devuelva

In [ ]:
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

search_wrapper = DuckDuckGoSearchAPIWrapper(
    region="es-es", # lenguaje de búsqueda: español
    max_results=5,  # número máximo de resultados a devolver
    backend="duckduckgo" # motor de búsqueda a utilizar (puede ser "duckduckgo" o "google")
)

search = DuckDuckGoSearchRun(
    api_wrapper=search_wrapper
)

-----       

## 3.- 🤖 Nuestro primer agente.    

Un ***agente*** es (según *Franklin and Graesser* (1997)):   

*"An autonomous agent is a system situated within and a part of an environment that senses that environment and acts on it, over time, in pursuit of its own agenda."*     
*"... un sistema situado dentro de un entorno, del que forma parte, y que es capaz de percibir dicho entorno y actuar sobre él a lo largo del tiempo, con el fin de alcanzar sus propios objetivos"*   

Esta definición ha sentado las bases para arquitecturas que integran la detección, la planificación, la ejecución y el aprendizaje. 

Formalmente, un agente también se puede describir como una tupla:

**A = ⟨E, P, R, A, U⟩**

donde:

**E — Entorno (Environment)**: el espacio de estados en el que opera el agente.     
**P — Percepciones**: las entradas que el agente recibe del entorno en cada instante.      
**R — Razonamiento / Política**: la función que mapea percepciones (o historiales de percepciones) a acciones.     
**A — Acciones**: el conjunto de operaciones que el agente puede ejecutar sobre el entorno.     
**U — Utilidad / Objetivo**: la métrica que el agente trata de optimizar.      

### Distinción clave respecto a un modelo de lenguaje simple    

Un LLM por sí solo es un transformador de texto (entrada → salida). Se convierte en agente cuando se le añade un bucle de razonamiento-acción, como el patrón **ReAct** (Reasoning + Acting):   


<img src="../img/ReAct.webp" style="display: block; margin: auto;">    


En resumen, tenemos este ciclo:   

```
Observación → Razonamiento → Acción → Nueva observación → …
```

Este ciclo se repite hasta que el agente considera la tarea completada o alcanza un límite de iteraciones.
        
        


Seguidamente, pasamos a implementar un agente sencillo, usando para ello el modelo `gpt-oss-20b` (modelo ligero-medio - ***20,9 Billones / 20.900 millones de parámetros*** - diseñado para ejecutarse en local), que soporta más de 14 idiomas (evaluados por Open AI) además de disponer de capacidades agénticas, soportando integración con herramientas externas para tareas complejas:

In [11]:
# Implementación de un agente sencillo sin herramientas, conectado a un modelo de Hugging Face


# --- Inicializar LLM desde HuggingFace ---
# llm_endpoint = HuggingFaceEndpoint(
#     repo_id="Qwen/Qwen3-4B-Instruct-2507",       
#     task="text-generation",
#     max_new_tokens=256,
#     temperature=0.1, # ajusta la creatividad de las respuestas (0.1 es bastante conservador y hará que las respuestas sean más precisas y menos creativas)
# )


# llm = ChatHuggingFace(llm=llm_endpoint) # creamos una instancia del modelo de lenguaje utilizando el endpoint configurado


# --- Inicializar LLM desde Groq ---
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024,
)



# --- Crear el agente ---
agent = create_agent(
    model=llm,
)   

In [12]:
# Ejemplo de uso del agente
pregunta = "¿Cuál es la capital de España?"
response = agent.invoke({
    "messages": [("user", pregunta)]
})
print("Respuesta del agente (Completa):", response) # mostramos la respuesta completa del agente, que incluye el mensaje del agente y la respuesta a la pregunta.


Respuesta del agente (Completa): {'messages': [HumanMessage(content='¿Cuál es la capital de España?', additional_kwargs={}, response_metadata={}, id='77145f9f-3b04-4d7d-a5bf-0405951d5769'), AIMessage(content='La capital de España es **Madrid**.', additional_kwargs={'reasoning_content': 'The user asks in Spanish: "¿Cuál es la capital de España?" The answer: Madrid. Provide in Spanish.'}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 79, 'total_tokens': 122, 'completion_time': 0.044459768, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.003896559, 'prompt_tokens_details': None, 'queue_time': 0.08608947, 'total_time': 0.048356327}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a198fb-3bbf-7286-b0a2-10d2ae754bb4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output

In [13]:
# Renderizamos la respuesta del agente, accediendo al último mensaje de la conversación y mostrando su contenido
from IPython.display import Markdown, display # importamos las funciones necesarias para mostrar la respuesta en formato Markdown
display(Markdown(f"**Respuesta del agente:** {response['messages'][-1].content}")) # mostramos el contenido del último mensaje del agente, que es la respuesta a la pregunta.

**Respuesta del agente:** La capital de España es **Madrid**.

Como se puede comprobar, el uso de un agente sin ningún tipo de 'accesorio'/herramienta no va más allá de la consulta realizada a un LLM utilizando dicho agente como mera 'interfaz' de comunicación con el LLM en cuestión.

En el próximo paso vamos a proporcionarle verdadera utilidad al agente, pasando a incorporarle la primera herramienta.       

----  

## 4.- 🔧 Definir e implementar nuestra primera `TOOL`  - (Patrón `Tool Use`)

El patrón "***Tool Using***" es una **inversión del control**. En otras palabras, mientras que en el anterior paradigma (*Chatbot*), el usuario pregunta y el modelo predice la siguiente palabra basándose en su entrenamiento (información congelada en el pasado), en este nuevo paradigma(*Tool Use*) el usuario pregunta y el modelo analiza si tiene la capacidad de responder.     

Si *detecta que le falta información o capacidad de cómputo*, no responde al usuario; en su lugar, *solicita ejecutar una función específica* de un entorno de programación.
    
Para que esto funcione, ocurre un proceso "*invisible*" de tres pasos (**HANDSHAKE**):

**1.- Declaración de Capacidades**: Al iniciar el chat, le enviamos al modelo un "Manual de Instrucciones" (Schemas). Le decimos: "Mira, no sé qué me va a pedir el usuario, pero aquí tienes 3 herramientas: una calculadora, un buscador web y un lector de disco. Úsalas si las necesitas."

**2.- Detección de Intención Semántica**: Cuando el usuario dice "Mi PC va lento", el modelo no busca la palabra "RAM" en su memoria. Entiende semánticamente que "lentitud" suele correlacionarse con "recursos del sistema" y decide usar la herramienta *`get_memory_usage`*.

**3.- Inyección de Realidad**: El resultado de la herramienta (ej. "RAM: 99% ocupada") se convierte en parte del contexto del modelo. Ahora, el modelo "sabe" algo que no estaba en su entrenamiento original.         

<img src="../img/agentic-ai-tool-use-pattern.jpg" style="display: block; margin: auto;">

Sí trasladamos esta pregunta a un chatbot al estilo de ChatGPT web, nos dirá probablemente que no tiene acceso a nuestro ordenador y, por tanto, no puede darnos una respuesta precisa. Si le preguntamos a nuestro agente, que está 'armado' con esta supuesta tool, este seguirá el siguiente flujo:   

1.- *Entiende la intención*: El usuario quiere un dato métrico actual.

2.- *Selecciona la herramienta*: De su caja de herramientas disponibles, elige `get_ram_metrics()`.

3.- *Ejecuta código*: El framework invoca la función Python localmente.

4.- *Interpreta el resultado*: Recibe `{ "free": "8.4 GB", "percent": 45.0 }`.

5.- *Responde*: `"Tienes 8.4 GB de RAM libre, el sistema está saludable."`

--- 

En este bloque de código crearemos la primera herramienta (`Tool`). En este caso, se trata de una función que es capaz de evaluar/realizar cálculos matemáticos usando para ello la función `sympify`de la librería `sympy`, que transforma cadenas de texto en expresiones matemáticas.   

También tenemos definida otra `Tool`para que se realicen búsquedas de información en Internet, en este caso usando el buscador DuckDuckGo, a través de su API.   

Cuando se implementa una `Tool` se deben considerar tres elementos o partes fundamentales:   

1.- El **nombre de la función**, que será el que utilice el agente para identificar y llamar a la `tool` en cuestión. Debemos usar nombres que sean lo suficientemente descriptivos para tener claro que es lo que hace.   
2.- El **docstring**. Este es, sin duda, el elemento más importante en la creación de una `tool`. El agente va a leer ese texto para decidir si debe usar o no la `tool`en cuestión y como debe ser su *signatura* para invocarla. Si el **docstring** está incompleto o no describe correctamente la utilidad de la `tool`, el agente no tendrá claro cuándo y/o cómo debe usarla.    
3.- La **lógica** de la función implementada: Es el código que se ejecuta cuando el agente decide usar la `tool`. Debe ***devolver siempre un texto (`str`)***, porque el agente necesita leer el resultado para continuar razonando.   

En conclusión, tenemos un esquema como este:    
```python
    @tool
    def nombre_descriptivo(parámetros):    # ← parte 1: nombre
        """
        Qué hace la tool.                  # ← parte 2: docstring
        Args: qué parámetros recibe.             (el agente lee esto)
        Returns: qué devuelve.
        """
        # lógica de la función            # ← parte 3: código
        return "resultado como texto"

```

#### El decorador **@tool**
Un decorador en Python es algo que se coloca encima de una función para añadirle funcionalidad extra sin modificar su código interno.
En este caso, **@tool** transforma una función Python normal en una herramienta que el agente puede usar. Sin él, el agente no sabría que esa función existe ni podría llamarla.       

A continuación, pasamos a implementar el código de las dos primeras `tools` de nuestro agente:

In [22]:
# --- Definir herramientas ---
@tool
def calculadora(expresion: str) -> str:
    """Evalúa expresiones matemáticas de forma segura. Ejemplo: 'sqrt(144)' o '2**10'"""
    try:
        resultado = sympify(expresion)
        return str(resultado)
    except Exception:
        return "Expresión matemática no válida"

search = DuckDuckGoSearchRun()

@tool
def busqueda_web(consulta: str) -> str:
    """Busca información actual en Internet."""
    try:
        return search.run(consulta)
    except Exception as e:
        return f"No se pudo realizar la búsqueda web: {e}"

### 📝 Ejercicio 1 - Implementación de una herramienta de consulta del clima de una ciudad
***Enunciado***   

Se pide implementar una tool llamada **clima** que permita a un agente conversacional nos devuelva el clima de una ciudad determinada.    

La **tool** debe:

- Recibir la ciudad cuyo clima  se quiere consultar
- Devolver la información sobre el tiempo/clima actual

Una vez implementada la tool, incorporarla al agente que estamos creando y que la use para responder preguntas como:

- *"Dime que tiempo hace en Valladolid"*

In [15]:
## Completa el siguiente código ##

# -----------------------------------------------
# TOOL: consulta_clima
# -----------------------------------------------

import requests

@tool
def clima(ciudad: str) -> str:
    """Consulta [completa_código] actual para una [completa_codigo]."""  #-- Completa el docstring con la descripción de la herramienta --
    
    # En la siguiente línea se realiza la consulta vía API a `wttr.in` para obtener el clima de la ciudad especificada. 
    # El parámetro `format=3` devuelve un resumen conciso del clima, incluyendo la temperatura y las condiciones meteorológicas actuales.
    url = f"https://wttr.in/{ciudad}?format=3"  
    
    response = requests.get(url)
    
    return response.text #-- Completa el return para devolver la respuesta de la consulta al clima --

<!-- Desplegable -->
<details>
<summary>💡 Solución (pulsa para mostrar)</summary>    

```python
# -----------------------------------------------
# TOOL: consulta_clima
# -----------------------------------------------

import requests

@tool
def clima(ciudad: str) -> str:
    """Consulta el clima actual para una ciudad."""
    
    url = f"https://wttr.in/{ciudad}?format=3"
    
    response = requests.get(url)
    
    return response.text 

```
</details>

### 📝 Ejercicio 2 - Implementación de una herramienta de conversión de unidades de temperatura (ºCelsius a ºFahrenheit)
***Enunciado***   

Se pide implementar una tool llamada **conversion_t** que permita a un agente conversacional transformar ºCelsius a ºFahrenheit.    

La **tool** debe:

- Recibir la temperatura en grados Celsius
- Devolver la temperatura convertida en grados Fahrenheit.    

Una vez implementada la tool, incorporarla al agente que estamos creando y que la use para responder preguntas como:

- *"¿Cuántos grados Fahrenheit son 18 ºC?"*

In [16]:
## Completa el siguiente código ##

# -----------------------------------------------
# TOOL: conversion_t
# -----------------------------------------------


@tool
def conversion_t(grados: str) -> str:
    """Convierte una temperatura de Celsius a Fahrenheit."""
    
    # 💡 PISTA: convierte el argumento a float y aplica la fórmula (°C × 9/5) + 32
    celsius    = float(grados)
    fahrenheit = (celsius * 9/5) + 32
    
    return f"{celsius}°C equivale a {fahrenheit:.1f}°F"

<!-- Desplegable -->
<details>
<summary>💡 Solución (pulsa para mostrar)</summary>    

```python
# -----------------------------------------------
# TOOL: conversion_t
# -----------------------------------------------

@tool
def conversion_t(grados: str) -> str:
    """Convierte una temperatura de Celsius a Fahrenheit."""
    
    # 💡 PISTA: convierte el argumento a float y aplica la fórmula (°C × 9/5) + 32
    celsius    = float(grados)
    fahrenheit = (celsius * 9/5) + 32
    
    return f"{celsius}°C equivale a {fahrenheit:.1f}°F"

```
</details>

En la siguiente línea de código se establece la lista de herramientas para que el agente pueda utilizarla en sus razonamientos y respuestas.

In [17]:
tools = [calculadora, busqueda_web, clima, conversion_t]

<!-- Nota -->
<div style="background-color: #fff3cd; color: #000000 ;padding: 15px; border-radius: 5px;">
<h5>⚠️ Una <b>tool</b> no tiene por qué hacer cálculos: simplemente <i>debe devolver información útil que el agente no podría obtener por sí solo</i> de forma fiable.</h5>
</div>

## 5.- 🧩 "Conectar" con el LLM que se utilizará como "motor" de nuestro Agente.   

En esta ocasión, a través de la API de GROQ, establecemos la conexión con el LLM escogido (`GPT-OSS-120B`).    

Es necesario disponer (o crear, si no se tiene) una cuenta en GROQ y gestionar la creación de un API token que, para este caso, solo necesita ser de tipo lectura (READ). Este se debe indicar en la linea indicada en el código.

In [19]:
# --- Inicializar LLM desde HuggingFace ---
# llm_endpoint = HuggingFaceEndpoint(
#     repo_id="Qwen/Qwen3-32B",  # cambiamos a un modelo más grande para mejorar la calidad de las respuestas       
#     task="text-generation",
#     max_new_tokens=1024,
#     temperature=0.1,
# )

# llm = ChatHuggingFace(llm=llm_endpoint)


# --- Inicializar LLM desde Groq ---  Son modelos conversacionales de gran tamaño, optimizados para generar respuestas coherentes y contextualmente relevantes en conversaciones.
llm = ChatGroq(
    model="openai/gpt-oss-120b", # cambiamos a un modelo más grande para mejorar la calidad de las respuestas 
    temperature=0.1,
    max_tokens=1024,
)

agent = create_agent(
    model=llm,
    tools=tools,
    # ajustamos el prompt para que el agente use las herramientas de manera secuencial y espere los resultados antes de continuar con la siguiente herramienta.
    system_prompt="Llama a las herramientas de forma secuencial cuando sea necesario. Espera el resultado de cada herramienta antes de llamar a la siguiente.",
)

## 6.- 🔍 Lanzar las consultas.    

En el siguiente bloque, utilizando la función `agent.invoke()` se lanzan las consultas a ejecutar por el agente. Cada consulta es una cadena de texto plano con la pregunta/petición realizada en lenguaje natural.

In [23]:
# --- Ejecutar consulta sobre cálculo numérico y búsqueda de noticias ---
response = agent.invoke({
    "messages": [("user", "¿Cuál es la raíz cuadrada de 144 y puedes buscar noticias recientes sobre ese número?")]
})

In [24]:
print(response)

{'messages': [HumanMessage(content='¿Cuál es la raíz cuadrada de 144 y puedes buscar noticias recientes sobre ese número?', additional_kwargs={}, response_metadata={}, id='cc137d66-1583-44f8-8180-e79a52b34c64'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the square root of 144 and can you search recent news about that number?" So we need to compute sqrt(144) = 12. Then search web for recent news about the number 12. Use calculadora tool for sqrt, then busqueda_web for news about "12". Probably query "news about number 12" or "12 news". We\'ll do two calls sequentially.', 'tool_calls': [{'id': 'fc_1ad01570-d7e7-4458-9fa3-c61d1372882c', 'function': {'arguments': '{"expresion":"sqrt(144)"}', 'name': 'calculadora'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 270, 'total_tokens': 389, 'completion_time': 0.245524912, 'completion_tokens_details': {'reasoning_tokens': 87}, 'prompt_time': 0.011

In [25]:
display(Markdown(response['messages'][-1].content)) # mostramos la respuesta del agente renderizada

**Respuesta**

- **Raíz cuadrada de 144:** 12  

- **Noticias recientes relacionadas con el número 12**

  1. **“Número 12” – Diario digital del Club Atlético Boca Juniors**  
     - Es el medio oficial del club que publica cada día noticias sobre el equipo, resultados, entrevistas y contenido multimedia.  
     - En las últimas horas se han difundido varias notas, entre ellas:  
       - *“Boca se prepara para el clásico contra River”* (análisis táctico y alineación provisional).  
       - *“Entrevista a Rodrigo Aquino: la visión del nuevo entrenador”*.  
       - *“Resumen del entrenamiento del lunes: los goles olímpicos que marcaron la diferencia”*.

  2. **Página 12 – Diario argentino de información y opinión**  
     - Aunque el nombre incluye el número 12, es un periódico de alcance nacional que cubre política, economía, cultura y deportes.  
     - Algunas de las publicaciones más recientes (últimos días) son:  
       - *“La ONU advierte que 770 millones de personas viven a menos de cinco metros sobre la pleamar”* (artículo sobre cambio climático).  
       - *“Debate en el Congreso: propuestas para financiar la adaptación al clima”*.  
       - *“Análisis de la crisis económica en Argentina: inflación y salarios”*.

  3. **Otros usos recientes del número 12 en la prensa**  
     - **Eventos y sorteos:** varios medios anunciaron sorteos de “12 pares de entradas de cine” para atraer al público durante la temporada de lluvias.  
     - **Fechas destacadas:** el 12 de junio 2024 y el 12 de septiembre 2024 aparecieron en titulares de noticias internacionales (por ejemplo, incendios forestales en California y giras políticas en EE. UU.).  

**Resumen:** La raíz cuadrada de 144 es 12. En la actualidad, el número 12 aparece con mayor frecuencia como parte del nombre de medios de comunicación argentinos — *Número 12* (Boca Juniors) y *Página 12* — que están publicando noticias de deporte, política y medio ambiente. Además, el número se usa en promociones y en la cobertura de fechas relevantes (12 de junio, 12 de septiembre) en la prensa internacional.

In [26]:
# --- Ejecutar  consulta sobre clima ---
response = agent.invoke({
    "messages": [("user", "¿Que tiempo hace en Valladolid y convierte esa temperatura a Fahrenheit?")]
})
display(Markdown(response['messages'][-1].content)) 

En Valladolid está soleado con una temperatura de **7 °C**, lo que equivale a **44.6 °F**.

In [27]:
# --- Visualizar lista de mensajes completa para ver el proceso de pensamiento ---
for msg in response['messages']:
    tipo = type(msg).__name__

    if tipo == "HumanMessage":
        print(f"👤 USUARIO: {msg.content}\n")

    elif tipo == "AIMessage":
        print(f"🤖 IA:")
        if msg.content:
            display(Markdown(msg.content))
        if msg.tool_calls:
            print("  🔧 Herramientas llamadas:")
            for tc in msg.tool_calls:
                print(f"     - {tc['name']}({tc['args']})")
        tokens = msg.usage_metadata
        if tokens:
            print(f"  📊 Tokens: entrada={tokens['input_tokens']}, salida={tokens['output_tokens']}, total={tokens['total_tokens']}")
        print()

    elif tipo == "ToolMessage":
        print(f"  🛠️  TOOL [{msg.name}]: {msg.content}\n")

👤 USUARIO: ¿Que tiempo hace en Valladolid y convierte esa temperatura a Fahrenheit?

🤖 IA:
  🔧 Herramientas llamadas:
     - clima({'ciudad': 'Valladolid'})
  📊 Tokens: entrada=264, salida=98, total=362

  🛠️  TOOL [clima]: Valladolid: ☀️  +7°C


🤖 IA:
  🔧 Herramientas llamadas:
     - conversion_t({'grados': '7'})
  📊 Tokens: entrada=305, salida=41, total=346

  🛠️  TOOL [conversion_t]: 7.0°C equivale a 44.6°F

🤖 IA:


En Valladolid está soleado con una temperatura de **7 °C**, lo que equivale a **44.6 °F**.

  📊 Tokens: entrada=345, salida=31, total=376



<!-- Nota -->
<div style="background-color: #fff3cd; color: #000000 ;padding: 15px; border-radius: 5px;">
⚠️ El consumo de tokens que se muestra es por el siguiente motivo:
Las herramientas no se ejecutan dentro del modelo, sino fuera de él. El modelo solo genera y recibe texto. El flujo real que tenemos es el siguiente:      

<img src="./images/flujo_tokens_agente.png" width="600" style="display: block; margin: auto;">

🚨 Por tanto, cada vez que el modelo necesita "leer" o "escribir" algo, **hay consumo de tokens**, y los ***resultados de las herramientas cuentan como texto de entrada en la siguiente llamada***
</div>


Hasta aquí la creación de nuestro primer agente. Como se ha comprobado, es relativamente sencillo implementar un agente básico y comprobar su correcto funcionamiento.     

Nuestro paso siguiente es configurar un ***historial de conversaciones***, para poder disponer de toda la lista de mensajes intercambiados entre el usuario y el agente a lo largo de una misma sesión.      

----

## 7.- 👥💬 Configurando el historial de conversaciones.     

Para configurar el historial de conversaciones, en este caso, vamos a importar los módulos `HumanMessage` y `AIMessage`del módulo de mensages principal de LangChain (`langchain.core_messages`).  El proceso, paso a paso, es como sigue: 
- Primero se configura una variable para almacenar todos los mensajes, comunmente denominada `message_history`.
- A continuación, definimos una nueva consulta, con una nueva pregunta sin información contextual adicional y con valores distintos para un nuevo cálculo.
- Se invoca al agente, ahora pasando tanto la nueva consulta como la variable que almacena todos los mensajes (historial de mensajes), todo ello dentro de un diccionario.   
- Se filtran únicamente los mensajes relevantes de la respuesta del agente (Usamos una "comprehension list" para seleccionar las instancias HumanMessage y AIMessage que disponen de contenido real). Al aplicar el metodo `strip()`se eliminan los espacios en blanco finales.   
- Finalmente, damos formato e imprimimos la conversación extraida del contenido de los mensajes, con cada mensaje etiquetado usando su nombre de clase correspondiente. Se observa que el `user_input`es ahora la nueva consulta mientras que el `agent_output`se corresponde con la consulta completa, cosa que resulta útil para depuración.

### 7.1 - Importamos nuevas clases necesarias.   

Importamos las clases `HumanMessage` y `AIMessage` para representar mensajes de entrada del usuario y respuestas del agente respectivamente

In [28]:
from langchain_core.messages import HumanMessage, AIMessage 

### 7.2 - Extracción del historial.   

Extraemos el historial de mensajes de la respuesta del agente

In [29]:
message_history = response["messages"]

### 7.3 - Definición de nuevas consultas.   

Se define una nueva consulta para el agente.

In [30]:
new_query = "¿Cuál es el área de un rectángulo con base 8 y altura 2?"

### 7.4 - Ejecución de la consulta.   

Lanzamos al agente pasando el historial de mensajes junto con la nueva consulta, lo que permite al agente mantener el contexto de la conversación y proporcionar una respuesta coherente basada en toda la información disponible.

In [31]:
messages = agent.invoke({"messages": message_history + [("user", new_query)]})

### 7.5 - Filtrado de mensajes.     

Se filtra el historial de mensajes para quedarnos solo con aquellos que son respuestas del agente (AIMessage) o entradas del usuario (HumanMessage), lo que nos permite centrarnos en la interacción relevante entre el usuario y el agente.

In [32]:
filtered_messages = [msg for msg in messages["messages"] 
                     if isinstance(msg, (HumanMessage, AIMessage))
                     and msg.content.strip()] 

### 7.6 - Elaboración de la salida (output).

Se construye un diccionario de salida que incluye la nueva consulta del usuario y una lista de las respuestas relevantes del agente, formateadas para mostrar claramente el tipo de mensaje (HumanMessage o AIMessage) junto con su contenido.

In [33]:
output ={
    "user_input": new_query,
    "agent_output": [f"{msg.__class__.__name__}: {msg.content}" for msg in filtered_messages]
}

Se formatea el output y se presenta el mensaje obtenido en markdown.

In [35]:
md = f"### 🧑 Consulta del Usuario\n{output['user_input']}\n\n### 🤖 Respuesta del Agente\n"
for line in output['agent_output']:

    # Convertir delimitadores LaTeX
    line = line.replace(r"\[", "$$")
    line = line.replace(r"\]", "$$")

    line = line.replace(r"\(", "$")
    line = line.replace(r"\)", "$")

    md += f"{line}\n\n"
    
display(Markdown(md)) 

### 🧑 Consulta del Usuario
¿Cuál es el área de un rectángulo con base 8 y altura 2?

### 🤖 Respuesta del Agente
HumanMessage: ¿Que tiempo hace en Valladolid y convierte esa temperatura a Fahrenheit?

AIMessage: En Valladolid está soleado con una temperatura de **7 °C**, lo que equivale a **44.6 °F**.

HumanMessage: ¿Cuál es el área de un rectángulo con base 8 y altura 2?

AIMessage: El área de un rectángulo se calcula multiplicando la base por la altura:

$$
\text{Área} = \text{base} \times \text{altura} = 8 \times 2 = 16
$$

Así que el área es **16 unidades cuadradas**.

